# Features

Run after `python -m similarity.preprocessing`.

In [1]:
# Work from the repo root, so `similarity` imports and the data/ paths resolve when opened from notebooks/
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [2]:
import numpy as np
import pandas as pd
import similarity.features as ft
from similarity.preprocessing import load_products

products = load_products()
f = ft.build_features(products)
len(f.ids)

30000

## text

In [3]:
pd.DataFrame({"name": products["name"].head(8), "prepared": products["name"].head(8).map(ft.prepare_text)})

,name,prepared
0,LA' Facon Cotton Kalamkari Handblock Saree Blo...,la' facon cotton kalamkari handblock saree blo...
1,Sf Jeans By Pantaloons Men's Plain Slim fit T-...,sf jeans by pantaloons men's plain slim fit ts...
2,LOVISTA Cotton Gota Patti Tassel Traditional P...,lovista cotton gota patti tassel traditional p...
3,People Men's Printed Regular fit T-Shirt,people men's printed regular fit tshirt
4,Monte Carlo Grey Solid Cotton Blend Polo Colla...,monte carlo grey solid cotton blend polo colla...
5,Forest Club | Gym Wear | Sports Shorts| Shorts...,forest club | gym wear | sports shorts| shorts...
6,PrintOctopus Graphic Printed T-Shirt for Men C...,printoctopus graphic printed tshirt for men ch...
7,Pepe Jeans Men's Solid Regular fit Casual Shirt,pepe jeans men's solid regular fit casual shirt


In [4]:
f.text.shape

(30000, 7412)

In [5]:
f.words[:30]

array(['00', '0005', '001', '002', '003', '004', '006', '009', '01',
       '0105', '011', '0110', '011h8b9627', '015', '02', '0210', '03',
       '031', '0310', '04', '045', '05', '06', '07', '07y', '0m',
       '0month', '0n', '0se', '10'], dtype=object)

In [6]:
(f.text.getnnz(axis=1) == 0).sum()

np.int64(2)

In [7]:
products.loc[f.text.getnnz(axis=1) == 0, "name"]

10043          Bauer Uncategorized
10431    SilverGrey-LadiesChuridar
Name: name, dtype: object

In [8]:
ft.top_words(f, 3)

people     0.701029
printed    0.375409
regular    0.350815
tshirt     0.300266
fit        0.299149
men        0.254763
dtype: float64

In [9]:
ft.top_words(f, 2)

tassel         0.406716
gota           0.396431
patti          0.376729
angrakha       0.374768
traditional    0.317026
pant           0.266389
salwar         0.261042
suit           0.245673
kurti          0.191194
with           0.151476
dtype: float64

In [10]:
rows = [3, 1, 7, 2]
pd.DataFrame((f.text[rows] @ f.text[rows].T).toarray(), index=products.loc[rows, "name"].str[:45]).round(2)

,0,1,2,3
name,,,,
People Men's Printed Regular fit T-Shirt,1.00,0.15,0.22,0.06
Sf Jeans By Pantaloons Men's Plain Slim fit T,0.15,1.00,0.24,0.00
Pepe Jeans Men's Solid Regular fit Casual Shi,0.22,0.24,1.00,0.00
LOVISTA Cotton Gota Patti Tassel Traditional,0.06,0.00,0.00,1.00


In [11]:
row = 3
scores = (f.text @ f.text[row].T).toarray().ravel()
top = np.argsort(scores)[::-1][:10]
products.loc[top, ["name", "child_category", "price"]].assign(text_score=scores[top].round(2))

,name,child_category,price,text_score
3277,People Men's Printed Regular fit T-Shirt,MensT_Shirts,199.0,1.00
8650,People Men's Printed Regular Fit T-Shirt,MensT_Shirts,277.0,1.00
24907,People Men's Printed Regular fit T-Shirt,MensT_Shirts,179.0,1.00
3,People Men's Printed Regular fit T-Shirt,MensT_Shirts,195.0,1.00
2970,People Men's Regular fit T-Shirt,MensT_Shirts,224.0,0.93
13815,People Men's Regular fit T-Shirt,MensT_Shirts,219.0,0.93
5289,People Men's Regular fit T-Shirt,MensT_Shirts,164.0,0.93
9099,People Men's Regular fit T-Shirt,MensT_Shirts,156.0,0.93
9787,People Men's Regular fit T-Shirt,MensT_Shirts,164.0,0.93
17189,People Men's Regular fit T-Shirt,MensT_Shirts,179.0,0.93


## brand

In [12]:
pd.DataFrame({"brand": products["brand"].head(10), "code": f.brand[:10]})

,brand,code
0,la' facon,0
1,None,-1
2,lovista,1
3,None,-1
4,None,-1
5,forest club,2
6,printoctopus,3
7,None,-1
8,carahere,4
9,peppermint,5


In [13]:
(f.brand == -1).sum(), f.brand.max() + 1

(np.int64(8392), np.int64(6336))

## category

In [14]:
pd.DataFrame({"category": products["child_category"].head(10), "code": f.category[:10]})

,category,code
0,WomensKurtasKurtis,0
1,MensT_Shirts,1
2,WomensKurtasKurtis,0
3,MensT_Shirts,1
4,MensTracksuits,2
5,MensShorts,3
6,MensT_Shirts,1
7,MensCasualShirts,4
8,None,-1
9,GirlsDressesJumpsuits,5


In [15]:
(f.category == -1).sum(), f.category.max() + 1

(np.int64(5149), np.int64(218))

In [16]:
((f.category == -1) & (f.browsenode == -1)).sum()

np.int64(99)

## gender

In [17]:
g = ft.genders(products)
g.value_counts(dropna=False)

women    12039
men      10390
boys      2731
None      2654
girls     2186
Name: count, dtype: int64

In [18]:
pd.DataFrame({"category": products["child_category"], "name": products["name"].str[:60], "gender": g}).sample(10, random_state=0)

,category,name,gender
8225,MensCasualShirts,United Colors of Benetton Men's Printed Slim F...,men
10794,WomensSportsShirtsTees,JUST RIDER Gym T-Shirts for Man & Women & Full...,women
9163,MensUnderwearVests,one8 by Virat Kohli Men's Vest,men
26591,BoysT_Shirts,Pepe Jeans Boy's Paisley Regular fit T-Shirt,boys
6631,None,Jesus Hawaiian Floral Snapback Christian Hat Cap,None
14161,MensFormalShirts,Peter England Men's Checkered Slim fit Formal ...,men
252,WomensJackets,TEXCO Winter Cotton Polyster Fleece Hooded Jacket,women
2566,BoysT_Shirts,Batman Boy's Regular fit T-Shirt,boys
3478,MensFormalShirts,"Men's Party Shirt,Green",men
2917,BoysShirts,Allen Solly Junior Boys' Shirt,boys


In [19]:
pd.crosstab(products["child_category"].notna(), g.notna())

col_0,False,True
child_category,,
False,2594,2555
True,60,24791


In [20]:
from_category = products["child_category"].map(lambda c: ft.derive_gender(c, ""))
from_title = products["name"].map(lambda n: ft.derive_gender(None, n))
both = from_category.notna() & from_title.notna()
pd.crosstab(from_category[both], from_title[both])

name,boys,girls,men,women
child_category,,,,
boys,2024,7,26,2
girls,8,1386,3,19
men,18,7,7922,37
women,6,32,21,8942


In [21]:
(from_category[both] == from_title[both]).mean()

np.float64(0.990909090909091)

In [22]:
(f.gender == -1).sum()

np.int64(2654)

## colour

In [23]:
colours = products["colours"].dropna()
changed = colours[colours.map(ft.normalise_colours) != colours]
pd.DataFrame({"before": changed, "after": changed.map(ft.normalise_colours)}).head(10)

,before,after
22,"[black, blackk, blue, bottle green, gray, gree...","[black, blackk, blue, bottle green, green, gre..."
53,"[blue, navy, olive, orange, rani, strawberry]","[blue, navy blue, olive, orange, rani, strawbe..."
75,"[beige, black, camel, dark brown, dark peach, ...","[beige, black, camel, dark brown, dark peach, ..."
91,"[golden, light maroon, light wine, navy blue, ...","[gold, light maroon, light wine, navy blue, pe..."
101,"[black, caribbean, ceil blue, chocolate, eggpl...","[black, caribbean, ceil blue, chocolate, eggpl..."
237,"[blue, blue/yellow, coral, coral/print, dark b...","[blue, blue/yellow, coral, coral/print, dark b..."
256,"[dark grey, off-white]","[dark grey, off white]"
295,"[black, multi-colored, multicolor, red]","[black, multi-colored, multicolour, red]"
326,"[black, multicolor, red, red-black]","[black, multicolour, red, red-black]"
329,"[grey, multi coloured, multi-coloured, multico...","[grey, multi coloured, multi-coloured, multico..."


In [24]:
len(changed)

733

In [25]:
f.colours.shape

(30000, 7819)

In [26]:
pd.Series(np.asarray(f.colours.sum(axis=0)).ravel(), index=f.colour_names).sort_values(ascending=False).head(15)

black         2380
red           1674
blue          1545
white         1247
navy blue     1124
pink          1044
green          942
yellow         934
grey           876
maroon         726
orange         610
brown          393
purple         369
royal blue     299
beige          289
dtype: int64

## price, weight, rating

In [27]:
pd.DataFrame({"price": f.price, "weight": f.weight, "rating": f.rating}).describe()

,price,weight,rating
count,27110.000000,6184.000000,30000.000000
mean,862.172397,257.845959,4.102887
std,964.223008,180.323286,0.271999
min,39.000000,9.070000,1.178391
25%,379.000000,150.000000,3.965564
50%,590.000000,240.000000,4.175367
75%,899.000000,349.000000,4.312806
max,9988.000000,4540.000000,4.996442


## duplicates

In [28]:
counts = np.bincount(f.asin)
(counts > 1).sum(), counts.max()

(np.int64(451), np.int64(3))